In [ ]:
import torch
import torchvision.transforms as transforms
import logging
import importlib
import numpy as np

from components import FL_sim
from components.other_utilities import models_to_train
import components.broadcast_components.reporting_utilities
import components.broadcast_components.broadcasting_process.WZ_broadcast
import components.other_utilities.datasets

importlib.reload(FL_sim)
importlib.reload(models_to_train)
importlib.reload(components.other_utilities.datasets)
importlib.reload(components.broadcast_components.broadcasting_process.WZ_broadcast)
importlib.reload(components.broadcast_components.reporting_utilities)

from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN
from components.broadcast_components.broadcasting_process.WZ_broadcast import WZBroadcastProtocol
from components.broadcast_components.reporting_utilities import BroadcastReportingUtilities

torch.set_float32_matmul_precision('high')

In [ ]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]
#             )
#         ])
#     ) for s in ['train', 'val']]


dataset = [
    FasterSVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

dataset = [torch.utils.data.Subset(d, list(range(100))) for d in dataset]
for i in range(10):
    for d in dataset:
        d.dataset.labels[i]=i

In [ ]:
k = 5  # Number of workers
logging_disabled=True
post_training_report=True

k = 3
logging_disabled=True
post_training_report=False

batch_size = 5000
# batch_size /= 6 # more batches as samples
# batch_size /= 50 # imagenet
# batch_size /= 3 # resnet 50
batch_size = int(batch_size)

# model.args_for_f_on_grad['save_folder_path'] = \
#     f"experiments/exp_data/gradients_resnet/gradients_resnet_t{i}/"

broadcast_prot_base = WZBroadcastProtocol(k,'RNN', tau=5,
        train_sample_size=100_000, metric_report_flag=False, lr=1e-5, num_planes=3, bins_per_plane=4)
broadcast_prot = BroadcastReportingUtilities(broadcast_prot_base)

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.01,
            logging_disabled=logging_disabled, applied_on_grads_before_optim=None)

# model.load_state_dict(torch.load('experiments/exp_data/resnet18_svhn.pth', map_location='cpu'))

sim = FLSimulator(
    num_agents=k, communication_rounds=10, client_epochs_per_round=15,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, pl_model=model,
)

# pre-training global model before saving it
# sim.do_train_global_model_and_set_local_model(num_epochs=4)
# torch.save(sim.global_model.state_dict(), 'experiments/exp_data/resnet18_svhn.pth')

import warnings
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="Starting from v1.9.0, `tensorboardX` has been removed")
warnings.filterwarnings("ignore", message="The 'val_dataloader' does not have many workers which may be a bottleneck")

sim.run_simulation(post_training_report=post_training_report, broadcast_prot=broadcast_prot, pre_training_global_epochs=0, )